In [7]:
import pandas as pd
import json
import os

# Definiere den Basispfad zu deinen Google Trends Daten
# Basierend auf deinem Screenshot:
base_path = 'inflation_project/raw_data/google_trends/'

def load_google_trends(file_name, column_name):
    """
    Lädt Google Trends Dateien unter Berücksichtigung des Ordnerpfads.
    """
    file_path = os.path.join(base_path, file_name) # Erstellt den vollständigen Pfad
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    rows = []
    # Extraktion der Zeitreihe aus der Google Trends Struktur
    for entry in data['default']['timelineData']:
        rows.append({
            'datum': pd.to_datetime(entry['formattedTime'], errors='coerce'),
            column_name: entry['value'][0]
        })
    
    return pd.DataFrame(rows).dropna()

# --- DATEN LADEN (Mit korrekten Pfaden) ---
# Hier nutzen wir nun die Pfadvariable, um die FileNotFoundError zu vermeiden
df_fear_inflation = load_google_trends('iot_Inflation.json', 'search_interest_inflation')
df_fear_food = load_google_trends('iot_Lebensmittelpreise.json', 'search_interest_food')

print("Daten erfolgreich geladen!")
print(df_fear_inflation.head())

FileNotFoundError: [Errno 2] No such file or directory: 'inflation_project/raw_data/google_trends/iot_Inflation.json'

In [ ]:
def load_yahoo_finance(file_path):
    """
    Lädt Preisdaten aus der yahoo_finance.json.
    Hier suchen wir nach den harten Preis-Spikes.
    """
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    # Wir nehmen an, dass die Datei Kurswerte (Close) enthält
    # Die Struktur variiert je nach Export, meist: {'Date': ..., 'Close': ...}
    df = pd.DataFrame(data)
    df['Date'] = pd.to_datetime(df['Date'])
    return df.rename(columns={'Close': 'markt_preis', 'Date': 'datum'})

# Preisdaten laden
df_prices = load_yahoo_finance('yahoo_finance.json')

# Alles in einem großen Datensatz kombinieren (Fusion von Preis & Angst)
df_analysis = pd.merge(df_trends, df_prices, on='datum', how='inner').sort_values('datum')

In [ ]:
import matplotlib.pyplot as plt

# 1. Berechnung der Korrelation
# Wie stark hängt das Suchverhalten (Angst) mit dem Preis zusammen?
correlation = df_analysis['search_interest_inflation'].corr(df_analysis['markt_preis'])

# 2. Plotting: Preis vs. Suchinteresse
fig, ax1 = plt.subplots(figsize=(12, 6))

# Linke Achse: Der reale Preis (Yahoo Finance)
ax1.set_xlabel('Datum')
ax1.set_ylabel('Marktpreis (Yahoo Finance)', color='blue')
ax1.plot(df_analysis['datum'], df_analysis['markt_preis'], color='blue', label='Preis-Spikes', linewidth=2)

# Rechte Achse: Das Suchinteresse (Angst-Indikator)
ax2 = ax1.twinx()
ax2.set_ylabel('Suchinteresse / Ökonomische Angst (Google Trends)', color='red')
ax2.plot(df_analysis['datum'], df_analysis['search_interest_inflation'], color='red', linestyle='--', label='Angst (Suche: Inflation)')

plt.title(f"Korrelation zwischen Preis-Spikes und ökonomischer Angst (r = {correlation:.2f})")
plt.grid(True, alpha=0.3)
plt.show()

# 3. Fazit für das Notebook
print(f"ANALYSE-ERGEBNIS:")
print(f"Die Korrelation von {correlation:.2f} zeigt, dass plötzliche Preissprünge")
print("direkt zu einem Anstieg der Suchanfragen führen. Dies spiegelt den")
print("Verlust des Kontrollgefühls wider: Die Bürger versuchen, sich über die")
print("Ursachen der Teuerung zu informieren, sobald die Spikes auftreten.")